#### Importing required libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Data Loading

In this section, we load the required datasets for the analysis.

For the initial investigation, we start with the Orders dataset because it contains the delivery-related timestamps required to identify delayed orders.

In [3]:
orders_original=pd.read_csv("data/olist_orders_dataset.csv")
orders_original.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


#### Initial Dataset Inspection
The goal of this step is to understand structure of the data,identifying important columns,inspect data types on the high level and to identify missing values if any.

In [4]:
orders_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


#### Observations
- The dataset contains 99441 entries

#### Data cleaning

In [5]:
orders_original[['order_delivered_customer_date','order_estimated_delivery_date']]=orders_original[['order_delivered_customer_date','order_estimated_delivery_date']].apply(pd.to_datetime,axis=0)

In [6]:
orders_original[["order_delivered_customer_date","order_estimated_delivery_date"]].dtypes

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

#### Handling missing values

In [7]:
orders_original["order_delivered_customer_date"].isnull().sum()

np.int64(2965)

#### observation
- There are 2000 missing entries in the order_delivered_customer_date field.These likely represent specific order statuses, such as processed, shipped, cancelled, invoiced, or unrecorded orders.
- To determine whether an order was delivered on time or delayed, both the actual delivery date and the estimated delivery date must be available.
- Orders with missing delivery dates cannot be reliably classified as delayed or on-time. Therefore, these records will be excluded from the delivery delay analysis.

In [8]:
orders=orders_original.dropna(subset=["order_delivered_customer_date"])
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### conclusion
- After removing orders with missing delivery dates, 96,476 orders remained for analysis.
- These records contain both actual and estimated delivery dates, allowing delivery performance to be evaluated accurately.

#### Feature Engineering: Delayed Order Flag
To evaluate delivery performance, a new feature called `is_delayed` is created.

Business Rule:
- If the actual delivery date is later than the estimated delivery date, the order is classified as delayed.
-  Otherwise, the order is classified as on-time.

This feature converts raw delivery timestamps into a business-friendly metric that can be used throughout the analysis.


In [9]:
orders["is_delayed"]=orders["order_delivered_customer_date"]>orders["order_estimated_delivery_date"]

In [10]:
orders['is_delayed'].value_counts()

is_delayed
False    88649
True      7827
Name: count, dtype: int64

#### Observation

After creating the delayed order flag, the majority of orders were delivered on time.

However, 7,827 orders were delivered later than the estimated delivery date. To understand the scale of this issue, the delayed order percentage should be calculated rather than relying on raw counts alone.

#### KPI 1: Percentage of Delayed Orders

The delayed order rate measures the proportion of delivered orders that arrived later than the estimated delivery date.

Formula:

(Delayed Orders / Total Orders) × 100

This KPI helps quantify the scale of delivery performance issues within the marketplace.

In [11]:
(orders['is_delayed'].value_counts(normalize=True)*100).to_frame()

,proportion
is_delayed,
False,91.887101
True,8.112899


#### KPI Observation

Approximately 8% of delivered orders arrived later than the estimated delivery date, while around 92% were delivered on time.

This indicates that the overall delivery performance is relatively strong. However, delivery delays still affect a meaningful number of customers. Further analysis is required to determine whether delayed deliveries are associated with lower customer satisfaction and review scores.


##### Preparing Customer Satisfaction Data

The next step is to incorporate customer review data into the analysis.

Review scores will be used as a proxy for customer satisfaction. By linking review scores with delivery performance, we can evaluate whether delayed orders receive lower ratings than on-time orders.

In [12]:
reviews=pd.read_csv("data/olist_order_reviews_dataset.csv")

In [13]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [14]:
reviews.shape

(99224, 7)

In [15]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [16]:
# checking missing values in reviews dataset
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

#### review dataset observation

The reviews dataset contains 99,224 customer reviews.

For the current analysis, the most important fields are:

- order_id: Used to connect reviews table with orders table.
- review_score: Represents customer satisfaction.

No missing values were found in the review_score column, making it suitable for customer satisfaction analysis.

In [149]:
# Count how many orders are missing customer reviews (Expects 0 if all are reviewed)
len(orders.loc[~orders['order_id'].isin(reviews['order_id'])])
# 646 orders don't have customer review

646

In [150]:
# merging orders table and reviews table
orders_reviews=orders.merge(reviews,on="order_id",how="inner")

In [151]:
print(orders_reviews.shape[0])
print(orders['order_id'].nunique())
print(orders_reviews['order_id'].nunique())
# IMP observation there were 646 orders which was without review
# after merging there are 96359 records but it should be 95830 , lets inspect review table

96359
96476
95830


In [152]:
print(len(reviews['order_id']))
print(reviews['order_id'].nunique())
# order_id key in review table might have some duplicates,lets verify 

99224
98673


In [153]:
reviews['order_id'].duplicated().sum()
# there are 551 duplicate order_id,which means each 

np.int64(551)

In [156]:
reviews[reviews.duplicated(subset=['order_id'],keep=False)].\
    sort_values(by=['order_id']).head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06


In [167]:
# lets fix the reviews table
reviews[["review_creation_date","review_answer_timestamp"]]=reviews[["review_creation_date",\
    "review_answer_timestamp"]]\
    .apply(pd.to_datetime)

In [168]:
reviews_cleaned=reviews.sort_values(by=['order_id',"review_answer_timestamp"])

In [174]:
reviews_cleaned=reviews.drop_duplicates(subset=['order_id'],keep='last')

In [186]:
orders_reviews=orders.merge(reviews_cleaned,on='order_id',how='inner')

In [198]:
orders_reviews[['order_id','is_delayed','review_score']].head()

,order_id,is_delayed,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,False,4
1,53cdb2fc8bc7dce0b6741e2150273451,False,4
2,47770eb9100c2d0c44946d9cf07ec65d,False,5
3,949d5b44dbf5de918fe9c16f97b45f8a,False,5
4,ad21c59c0840e6cb83a9ceb5573f8159,False,5
